The Vision Transformer (ViT) is a transformer encoder model (BERT-like) pretrained on a large collection of images in a supervised fashion, namely ImageNet-21k, at a resolution of 224x224 pixels. Next, the model was fine-tuned on ImageNet (also referred to as ILSVRC2012), a dataset comprising 1 million images and 1,000 classes, also at resolution 224x224.
Images are presented to the model as a sequence of fixed-size patches (resolution 16x16), which are linearly embedded.

arXiv:2010.11929

Here transfer learning for Cifar-10 is applied

In [1]:
import keras
import keras_hub
import tensorflow as tf
import os

os.environ["KERAS_BACKEND"] = "tensorflow"

2025-09-26 13:54:13.427695: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-09-26 13:54:13.434138: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1758894853.441700 2235802 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1758894853.443950 2235802 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-09-26 13:54:13.452023: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

In [4]:
gpus = tf.config.list_physical_devices(device_type='GPU')
print(gpus)
if gpus:
  try:
    # Set memory growth to avoid DNN library initialization errors
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    # Optionally, restrict to only the first GPU
    tf.config.set_visible_devices(gpus[0], 'GPU')
    logical_gpus = tf.config.list_logical_devices('GPU')
    print(f"{len(gpus)} Physical GPUs, {len(logical_gpus)} Logical GPUs")
  except RuntimeError as e:
    print(e)

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]
2 Physical GPUs, 1 Logical GPUs


In [5]:
(training_images, training_labels) , \
(validation_images, validation_labels) = \
tf.keras.datasets.cifar10.load_data()

In [6]:
def preprocess_image_input(input_images):
  input_images = input_images.copy().astype('float32')
  output_ims = tf.keras.applications.imagenet_utils.preprocess_input(input_images, mode='tf')
  return output_ims

train_X = preprocess_image_input(training_images)
valid_X = preprocess_image_input(validation_images)

In [17]:
@keras.saving.register_keras_serializable()
class ResizeLayer(tf.keras.layers.Layer):
  def __init__(self, size, method='nearest', **kwargs):
    super(ResizeLayer, self).__init__(**kwargs)
    self.size = size
    self.method = method

  def call(self, inputs):
    return tf.image.resize(inputs, size=self.size, method=self.method)

@tf.keras.utils.register_keras_serializable()
class SelectCLS(tf.keras.layers.Layer):
    def call(self, inputs):
        # Prende solo il token [CLS]
        return inputs[:, 0, :]

backbone = keras_hub.models.Backbone.from_preset(
    "vit_base_patch16_224_imagenet21k"
)

preprocessor = keras_hub.models.ViTImageClassifierPreprocessor.from_preset(
    "vit_base_patch16_224_imagenet21k"
)

vit = keras_hub.models.ViTImageClassifier(
    backbone=backbone,
    num_classes=10,
    preprocessor=preprocessor,
)

vit.summary()
# adding a new classification head for CIFAR-10
#inputs = tf.keras.layers.Input(shape=(32, 32, 3))
##x = ResizeLayer(size=(224, 224))(inputs)   
#x = vit(x)            
#x = SelectCLS()(x) 
#outputs = tf.keras.layers.Dense(10, activation="softmax")(x)

#model = keras.Model(inputs, x)


# model.compile(
#   optimizer=keras.optimizers.Adam(1e-3),
#   loss="sparse_categorical_crossentropy",
#   metrics=["accuracy"]
# )

# model.summary()


Preprocessor: "vi_t_image_classifier_preprocessor"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                                                  ┃                                   Config ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ vi_t_image_converter (ViTImageConverter)                      │                   Image size: (224, 224) │
└───────────────────────────────────────────────────────────────┴──────────────────────────────────────────┘

Model: "vi_t_image_classifier_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                                  ┃ Output Shape                       ┃             Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer_12 (InputLayer)                   │ (None, 224, 224, 3)                │                   0 │
├───────────────────────────────────────────────┼────────────────────────────────────┼─────────────────────┤
│ vi_t_backbone (ViTBackbone)                   │ (None, 197, 768)                   │          85,798,656 │
├───────────────────────────────────────────────┼────────────────────────────────────┼─────────────────────┤
│ get_item_6 (GetItem)                          │ (None, 768)                        │                   0 │
├───────────────────────────────────────────────┼────────────────────────────────────┼─────────────────────┤
│ output_dropout (Dropout)                      │ (None, 768)                        │                   0 │
├───────────────────────────────────────────────┼────────────────────────────────────┼─────────────────────┤
│ predictions (Dense)                           │ (None, 10)                         │               7,690 │
└───────────────────────────────────────────────┴────────────────────────────────────┴─────────────────────┘

 Total params: 85,806,346 (327.33 MB)

 Trainable params: 85,806,346 (327.33 MB)

 Non-trainable params: 0 (0.00 B)

In [19]:
history = vit.fit(train_X, training_labels, 
                    validation_data=(valid_X, validation_labels), 
                    epochs=1)

2025-09-26 14:19:14.484806: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'copy_fusion_5', 608 bytes spill stores, 608 bytes spill loads
ptxas warning : Registers are spilled to local memory in function '__cuda_sm3x_div_rn_noftz_f32_slowpath', 52 bytes spill stores, 52 bytes spill loads
ptxas warning : Registers are spilled to local memory in function 'copy_fusion_1', 472 bytes spill stores, 472 bytes spill loads
ptxas warning : Registers are spilled to local memory in function '__cuda_sm3x_div_rn_noftz_f32_slowpath', 48 bytes spill stores, 48 bytes spill loads
ptxas warning : Registers are spilled to local memory in function 'copy_fusion_6', 608 bytes spill stores, 608 bytes spill loads
ptxas warning : Registers are spilled to local memory in function '__cuda_sm3x_div_rn_noftz_f32_slowpath', 52 bytes spill stores, 52 bytes spill loads
ptxas warning : Registers are spilled to local memory in fun

1562/1563 ━━━━━━━━━━━━━━━━━━━━ 0s 313ms/step - loss: 2.3089 - sparse_categorical_accuracy: 0.0985

2025-09-26 14:27:29.146537: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'copy_fusion_4', 608 bytes spill stores, 608 bytes spill loads
ptxas warning : Registers are spilled to local memory in function '__cuda_sm3x_div_rn_noftz_f32_slowpath', 52 bytes spill stores, 52 bytes spill loads
ptxas warning : Registers are spilled to local memory in function 'copy_fusion_1', 472 bytes spill stores, 472 bytes spill loads
ptxas warning : Registers are spilled to local memory in function '__cuda_sm3x_div_rn_noftz_f32_slowpath', 48 bytes spill stores, 48 bytes spill loads
ptxas warning : Registers are spilled to local memory in function 'copy_fusion_2', 608 bytes spill stores, 608 bytes spill loads
ptxas warning : Registers are spilled to local memory in function '__cuda_sm3x_div_rn_noftz_f32_slowpath', 52 bytes spill stores, 52 bytes spill loads
ptxas warning : Registers are spilled to local memory in fun

1563/1563 ━━━━━━━━━━━━━━━━━━━━ 548s 336ms/step - loss: 2.3089 - sparse_categorical_accuracy: 0.0985 - val_loss: 2.3070 - val_sparse_categorical_accuracy: 0.1000


In [20]:
vit.evaluate(validation_images, validation_labels)
model.evaluate(validation_images[:2000], validation_labels[:2000])

313/313 ━━━━━━━━━━━━━━━━━━━━ 26s 81ms/step - loss: 2.3011 - sparse_categorical_accuracy: 0.1460
63/63 ━━━━━━━━━━━━━━━━━━━━ 5s 80ms/step - accuracy: 0.1138 - loss: 10.5159


[10.519333839416504, 0.10949999839067459]

In [ ]:
MODEL_PATH = "../models_data/ViT-b_16p_224/"
os.makedirs(MODEL_PATH, exist_ok=True)
model.save(MODEL_PATH+'/ViT-b_16p_224_fp32_imagenet2012.keras')

In [ ]:
def representative_dataset():
  for images in tf.data.Dataset.from_tensor_slices(validation_images).batch(1).take(1000):
    yield [images]

loaded_model = keras.saving.load_model("../models_data/ViT-b_16p_224/ViT-b_16p_224_fp32_imagenet2012.keras")
converter = tf.lite.TFLiteConverter.from_keras_model(loaded_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT] 
converter.representative_dataset = representative_dataset

converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.uint8
converter.inference_output_type = tf.uint8
tflite_quant_model = converter.convert()
with open('../models_data/ViT-b_16p_224/ViT-b_16p_224_uint8_imagenet2012.tflite', 'wb') as f:
  f.write(tflite_quant_model)

converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8
tflite_quant_model = converter.convert()
with open('../models_data/ViT-b_16p_224/ViT-b_16p_224_int8_imagenet2012.tflite', 'wb') as f:
  f.write(tflite_quant_model)